# IEEE-CIS Fraud Detection: Evaluation

**Goal:** Analyze model performance and understand predictions

**Contents:**
- ROC and Precision-Recall curves
- Threshold analysis (F1 vs Cost optimization)
- Cost-benefit analysis
- Feature importance deep dive

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../data')
PROCESSED_DIR = DATA_DIR / 'processed'
MODELS_DIR = Path('../models')
REPORTS_DIR = Path('../reports')
REPORTS_DIR.mkdir(exist_ok=True)
FIG_DIR = Path('../figures/evaluation')
FIG_DIR.mkdir(exist_ok=True)

## 1. Load Model and Data

In [ ]:
# Load data
X_train = pd.read_parquet(PROCESSED_DIR / 'X_train.parquet')
y_train = pd.read_parquet(PROCESSED_DIR / 'y_train.parquet')['isFraud']

# Load models
final_model = joblib.load(MODELS_DIR / 'lgb_model.joblib')
cv_models = joblib.load(MODELS_DIR / 'lgb_cv_models.joblib')
cv_scores = pd.read_csv(MODELS_DIR / 'cv_scores.csv')

print(f"Data shape: {X_train.shape}")
print(f"CV Scores: {cv_scores['auc'].values}")
print(f"Mean AUC: {cv_scores['auc'].mean():.4f}")

## 2. Holdout Evaluation

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, average_precision_score

# Create holdout set (last 20% by time)
split_idx = int(len(X_train) * 0.8)
X_holdout = X_train.iloc[split_idx:]
y_holdout = y_train.iloc[split_idx:]

# Predict
y_pred_proba = final_model.predict(X_holdout)

print(f"Holdout size: {len(X_holdout):,}")
print(f"Holdout fraud rate: {y_holdout.mean():.2%}")
print(f"ROC-AUC: {roc_auc_score(y_holdout, y_pred_proba):.4f}")
print(f"PR-AUC: {average_precision_score(y_holdout, y_pred_proba):.4f}")

## 3. ROC and PR Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
fpr, tpr, thresholds_roc = roc_curve(y_holdout, y_pred_proba)
auc_score = roc_auc_score(y_holdout, y_pred_proba)

axes[0].plot(fpr, tpr, label=f'Model (AUC = {auc_score:.4f})', linewidth=2)
axes[0].plot([0, 1], [0, 1], 'k--', label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curve
precision, recall, thresholds_pr = precision_recall_curve(y_holdout, y_pred_proba)
pr_auc = average_precision_score(y_holdout, y_pred_proba)

axes[1].plot(recall, precision, label=f'Model (PR-AUC = {pr_auc:.4f})', linewidth=2)
axes[1].axhline(y=y_holdout.mean(), color='k', linestyle='--', label=f'Baseline ({y_holdout.mean():.2%})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.savefig(FIG_DIR / 'roc_pr_curves.png')
plt.show()

## 4. Threshold Analysis

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Analyze different thresholds
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]

print("Performance at different thresholds:\n")
for thresh in thresholds:
    y_pred = (y_pred_proba >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_holdout, y_pred).ravel()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"Threshold {thresh}:")
    print(f"  Precision: {precision:.2%} | Recall: {recall:.2%} | F1: {f1:.2%}")
    print(f"  TP: {tp:,} | FP: {fp:,} | FN: {fn:,} | TN: {tn:,}")
    print()

In [ ]:
# Find optimal threshold (maximize F1)
f1_scores = []
for thresh in np.arange(0.01, 0.99, 0.01):
    y_pred = (y_pred_proba >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_holdout, y_pred).ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    f1_scores.append((thresh, f1, precision, recall))

f1_df = pd.DataFrame(f1_scores, columns=['threshold', 'f1', 'precision', 'recall'])
best_thresh = f1_df.loc[f1_df['f1'].idxmax()]

print(f"Optimal threshold: {best_thresh['threshold']:.2f}")
print(f"  F1: {best_thresh['f1']:.2%}")
print(f"  Precision: {best_thresh['precision']:.2%}")
print(f"  Recall: {best_thresh['recall']:.2%}")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(f1_df['threshold'], f1_df['f1'], label='F1', linewidth=2)
ax.plot(f1_df['threshold'], f1_df['precision'], label='Precision', linewidth=2)
ax.plot(f1_df['threshold'], f1_df['recall'], label='Recall', linewidth=2)
ax.axvline(x=best_thresh['threshold'], color='red', linestyle='--', label=f"Optimal ({best_thresh['threshold']:.2f})")
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision, Recall, F1 vs Threshold')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / 'threshold_analysis.png')
plt.show()

## 5. Confusion Matrix

In [ ]:
# Use optimal threshold
y_pred_optimal = (y_pred_proba >= best_thresh['threshold']).astype(int)
cm = confusion_matrix(y_holdout, y_pred_optimal)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=ax,
            xticklabels=['Predicted Legit', 'Predicted Fraud'],
            yticklabels=['Actual Legit', 'Actual Fraud'])
ax.set_title(f'Confusion Matrix (Threshold = {best_thresh["threshold"]:.2f})')
fig.tight_layout()
fig.savefig(FIG_DIR / 'confusion_matrix.png')
plt.show()

print(classification_report(y_holdout, y_pred_optimal, target_names=['Legitimate', 'Fraud']))

## 6. Score Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Score distribution by class
ax.hist(y_pred_proba[y_holdout == 0], bins=50, alpha=0.5, label='Legitimate', density=True)
ax.hist(y_pred_proba[y_holdout == 1], bins=50, alpha=0.5, label='Fraud', density=True)
ax.axvline(x=best_thresh['threshold'], color='red', linestyle='--', label=f"Threshold ({best_thresh['threshold']:.2f})")
ax.set_xlabel('Predicted Probability')
ax.set_ylabel('Density')
ax.set_title('Score Distribution by Class')
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / 'score_distribution.png')
plt.show()

## 7. Cost-Benefit Analysis

Optimize threshold based on business costs, not just F1.

In [ ]:
# Define business costs
from dataclasses import dataclass

@dataclass
class CostMatrix:
    """Cost matrix for fraud detection outcomes."""
    false_negative_cost: float = 150.0  # Missed fraud (avg fraud amount)
    false_positive_cost: float = 10.0   # Blocked legitimate (friction cost)
    true_positive_cost: float = 5.0     # Caught fraud (review cost)
    true_negative_cost: float = 0.0     # Approved legitimate

costs = CostMatrix()
print(f"Cost of missed fraud (FN): ${costs.false_negative_cost}")
print(f"Cost of blocking legitimate (FP): ${costs.false_positive_cost}")
print(f"Cost of reviewing fraud (TP): ${costs.true_positive_cost}")

In [ ]:
# Calculate cost at each threshold
def calculate_cost(y_true, y_pred, costs):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    total_cost = (
        fn * costs.false_negative_cost +
        fp * costs.false_positive_cost +
        tp * costs.true_positive_cost +
        tn * costs.true_negative_cost
    )
    return total_cost, {'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp}

# Find cost-optimal threshold
cost_results = []
for thresh in np.arange(0.01, 0.99, 0.01):
    y_pred = (y_pred_proba >= thresh).astype(int)
    total_cost, counts = calculate_cost(y_holdout, y_pred, costs)
    
    # Baseline: no model (approve everything)
    baseline_cost = y_holdout.sum() * costs.false_negative_cost
    savings = baseline_cost - total_cost
    
    cost_results.append({
        'threshold': thresh,
        'total_cost': total_cost,
        'baseline_cost': baseline_cost,
        'savings': savings,
        **counts
    })

cost_df = pd.DataFrame(cost_results)
best_cost_thresh = cost_df.loc[cost_df['total_cost'].idxmin()]

print(f"Cost-optimal threshold: {best_cost_thresh['threshold']:.2f}")
print(f"  Total cost: ${best_cost_thresh['total_cost']:,.0f}")
print(f"  Baseline cost (no model): ${best_cost_thresh['baseline_cost']:,.0f}")
print(f"  Savings: ${best_cost_thresh['savings']:,.0f}")
print(f"\nCompare to F1-optimal threshold: {best_thresh['threshold']:.2f}")

In [ ]:
# Visualize cost vs threshold
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cost curve
axes[0].plot(cost_df['threshold'], cost_df['total_cost'], label='Total Cost', linewidth=2)
axes[0].axvline(x=best_cost_thresh['threshold'], color='green', linestyle='--', 
                label=f"Cost-optimal ({best_cost_thresh['threshold']:.2f})")
axes[0].axvline(x=best_thresh['threshold'], color='red', linestyle=':', 
                label=f"F1-optimal ({best_thresh['threshold']:.2f})")
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('Total Cost ($)')
axes[0].set_title('Total Cost vs Threshold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Savings curve
axes[1].plot(cost_df['threshold'], cost_df['savings'], label='Savings vs No Model', linewidth=2, color='green')
axes[1].axvline(x=best_cost_thresh['threshold'], color='green', linestyle='--')
axes[1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[1].fill_between(cost_df['threshold'], 0, cost_df['savings'], alpha=0.3, color='green')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Savings ($)')
axes[1].set_title('Savings vs No Model (Baseline)')
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / 'cost_analysis.png')
plt.show()

## 8. Feature Importance Analysis

In [ ]:
# Feature importance
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': final_model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

# Top and bottom features
print("Top 20 features:")
print(importance_df.head(20).to_string(index=False))

print("\nBottom 10 features (candidates for removal):")
print(importance_df.tail(10).to_string(index=False))

In [ ]:
# Cumulative importance
importance_df['cumulative'] = importance_df['importance'].cumsum() / importance_df['importance'].sum()

n_features_90 = (importance_df['cumulative'] <= 0.9).sum()
n_features_95 = (importance_df['cumulative'] <= 0.95).sum()

print(f"Features needed for 90% importance: {n_features_90} / {len(importance_df)}")
print(f"Features needed for 95% importance: {n_features_95} / {len(importance_df)}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(importance_df) + 1), importance_df['cumulative'].values, linewidth=2)
ax.axhline(y=0.9, color='orange', linestyle='--', label='90%')
ax.axhline(y=0.95, color='red', linestyle='--', label='95%')
ax.set_xlabel('Number of Features')
ax.set_ylabel('Cumulative Importance')
ax.set_title('Cumulative Feature Importance')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / 'cumulative_feature_importance.png')
plt.show()

## 9. Summary

In [ ]:
summary = f"""
=== Model Evaluation Summary ===

CV Performance:
  Mean AUC: {cv_scores['auc'].mean():.4f} (+/- {cv_scores['auc'].std():.4f})

Holdout Performance:
  ROC-AUC: {roc_auc_score(y_holdout, y_pred_proba):.4f}
  PR-AUC: {average_precision_score(y_holdout, y_pred_proba):.4f}

Threshold Optimization:
  F1-optimal: {best_thresh['threshold']:.2f} (F1={best_thresh['f1']:.2%})
  Cost-optimal: {best_cost_thresh['threshold']:.2f} (saves ${best_cost_thresh['savings']:,.0f})

Cost Analysis (at cost-optimal threshold):
  Baseline cost (no model): ${best_cost_thresh['baseline_cost']:,.0f}
  Model cost: ${best_cost_thresh['total_cost']:,.0f}
  Net savings: ${best_cost_thresh['savings']:,.0f}

Feature Analysis:
  Total features: {len(importance_df)}
  Features for 90% importance: {n_features_90}
  Top 5 features: {', '.join(importance_df.head(5)['feature'].tolist())}
"""

print(summary)

# Save summary
with open(REPORTS_DIR / 'evaluation_summary.txt', 'w') as f:
    f.write(summary)

In [ ]:
"""
kaggle competitions submit -c ieee-fraud-detection -f submission.csv -m "Message"
"""